# Phase 5 — J2塑性と材料点更新：答案Notebook

[本文](../docs/texts/phase5-j2-plasticity.md) / [学習ログ](../docs/learning-log.md)

## 進め方

6演習・100点。記述・数式・コード・保存出力をこのNotebookへまとめて提出する。
各問で予測を書いてから実装し、予測と異なった結果を説明する。
実装するのは微小ひずみ・3D・等方弾性・関連J2塑性・線形等方硬化の材料点モデル。
完成したソルバコードは配布しない。関数の分け方・内部実装・追加検証は自分で設計する。

コードセルの関数は `NotImplementedError` の答案欄であり、未実装時はチェックを「未実装」と表示する。
これは合格ではない。提出時はカーネルを再起動して全セル実行し、「未実装」を残さない。
自作の表・グラフ用セルは各問の直後へ自由に追加してよい。

| 問 | 内容 | 点数（記述／実装・検証） |
| --- | --- | --- |
| 1 | 応力不変量と弾性分解 | 15（7／8） |
| 2 | 流れ則・累積量・硬化係数 | 20（12／8） |
| 3 | 3D radial return | 25（8／17） |
| 4 | 反転負荷と履歴 | 15（5／10） |
| 5 | 回転・増分細分化・入力不変 | 15（5／10） |
| 6 | FEM・CAE結果レビュー | 10（10／0） |

必須依存はNumPy、描画にはMatplotlibを使う。応力・弾性係数はMPa、ひずみは無次元。
全テンソルは対称3×3配列。せん断ひずみはテンソル成分を入れる。
配布チェックは最低限の目印で、指定された自作検証の代わりにはならない。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

E, nu, sigma_y0, H = 210000.0, 0.3, 250.0, 1000.0
G = E / (2 * (1 + nu))
K = E / (3 * (1 - 2 * nu))
I = np.eye(3)
Z = np.zeros((3, 3))

def checkpoint(label, check):
    try:
        check()
    except NotImplementedError:
        print(f"未実装: {label}（検証未完了）")
    else:
        print(f"PASS: {label}（自作検証・記述も必要）")

print(f"G={G:.6f} MPa, K={K:.6f} MPa")

## 演習1 — 体積・偏差と降伏判定（15点）

本文1〜2節。$p$ は引張正の平均応力、$q$ はMises応力、$J_2=\mathbf s:\mathbf s/2$。

1. 一軸応力、純せん断、静水圧応力の $\mathbf s,J_2,q$ を導出し、純せん断の縮約で係数2が生じる理由を書く（7点）。
2. `invariants(sigma)` を実装する。返り値は `(p, s, J2, q)`。`s` は3×3配列、他はスカラー。`elastic_stress(eps, E, nu)` も実装する。入力を変更しない（5点）。
3. 配布チェックに加え、任意の対称応力への静水圧応力追加で $q$ が不変なことを自作テストする。比較したテンソルと差を表示する（3点）。

対称3×3・有限値を必須とし、異常形状・非対称・NaNを `ValueError` で拒否する。
弾性係数は有限スカラーで $E>0,-1<\nu<1/2$。対称性の許容誤差は自分で明記する。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

In [ ]:
def invariants(sigma):
    # TODO: 入力検証、平均・偏差・二重縮約
    raise NotImplementedError

def elastic_stress(eps, E, nu):
    # TODO: 体積成分と偏差成分への弾性応答
    raise NotImplementedError

def check_ex1():
    cases = [(np.diag([300., 0., 0.]), 300.),
             (np.array([[0., 100., 0.], [100., 0., 0.], [0., 0., 0.]]), np.sqrt(3)*100),
             (100.*I, 0.)]
    for stress, q_expected in cases:
        p, s, j2, q = invariants(stress)
        np.testing.assert_allclose(q, q_expected, atol=1e-9, rtol=1e-10)
        np.testing.assert_allclose(stress, p*I+s, atol=1e-9)
        np.testing.assert_allclose(np.trace(s), 0., atol=1e-9)
    np.testing.assert_allclose(elastic_stress(1e-4*I, E, nu), 52.5*I, atol=1e-9)

checkpoint("演習1の基準値", check_ex1)
# TODO: 自作検証と表示を追加

## 演習2 — 流れ則・相当塑性ひずみ・硬化（20点）

本文3〜4節。

1. $q^2=3\mathbf s:\mathbf s/2$ を微分して $\partial f/\partial\boldsymbol\sigma=3\mathbf s/(2q)$ を導く。トレースゼロと $\dot\alpha=\dot\lambda$ を示す。相補条件を使い、降伏面上の弾性除荷を説明する（8点）。
2. 一軸応力の単調引張について $E_t=EH/(E+H)$ を導出する。$E_t=1000$ MPaの測定曲線から $H$ を計算し、共通定数の $H=1000$ MPaとの違いを説明する（4点）。
3. `accumulate_alpha(dep_history)` を実装する。入力は各ステップの塑性ひずみ増分 `(m,3,3)`、出力は初期値0を含む `(m+1,)` の累積量。各増分内の流れ方向を一定とみなし、増分ノルムを加算する。空履歴 `(0,3,3)` は `[0.]` を返す（5点）。
4. $\mathbf A=\operatorname{diag}(1,-1/2,-1/2)$ とし、増分 $0.002\mathbf A,-0.002\mathbf A$ を順に与える。最終塑性ひずみテンソルのノルムと累積量を表で比較する。一般経路で等しくない理由と、$\boldsymbol\sigma:\dot{\boldsymbol\varepsilon}^p=q\dot\alpha$ の意味を書く（3点）。

ここは指定した塑性増分の幾何を調べる問題で、全ひずみから材料応答を求める問題は演習3から扱う。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

In [ ]:
def accumulate_alpha(dep_history):
    # TODO: 各塑性増分の sqrt(2/3 * dep:dep) を累積
    raise NotImplementedError

def check_ex2():
    A = np.diag([1., -0.5, -0.5])
    history = np.array([0.002*A, -0.002*A])
    result = accumulate_alpha(history)
    np.testing.assert_allclose(result, [0., 0.002, 0.004], atol=1e-12)
    np.testing.assert_allclose(accumulate_alpha(np.empty((0, 3, 3))), [0.])
    print("塑性増分の総和:", history.sum(axis=0), "累積量:", result[-1])

checkpoint("演習2の反転履歴", check_ex2)
# TODO: 硬化係数計算、表、入力検証を追加

## 演習3 — 3D材料点更新を設計する（25点）

本文5節。

1. 弾性試行応力から後退Eulerの流れ則へ進み、偏差応力の方向が変わらないことを示して $\Delta\alpha=f^{\mathrm{tr}}/(3G+H)$ を導く。平均応力まで縮めてはいけない理由も書く（8点）。
2. 次の関数を実装する（12点）。
   - 入力：全ひずみ**増分** `deps`、確定済み応力 `sigma_n`、塑性ひずみ `ep_n`、累積量 `alpha_n`、材料定数。
   - 出力：`(sigma_new, ep_new, alpha_new, dalpha)`。初めの二つは3×3配列、後ろはスカラー。
   - `alpha_n >= 0, sigma_y0 > 0, H >= 0`。全入力の有限性・形状・対称性を検査し、不正入力は `ValueError`。入力履歴は書き換えない。
   - 入力履歴と全ひずみの構成則上の整合性は呼出し側の責任。初期値はすべてゼロ。
   - 弾性分岐でゼロ除算を回避する。許容値を導入したら単位・スケールを答案へ書く。
3. 次を自作検証し、実測誤差を表示する（5点）：弾性負荷、静水圧負荷、純せん断での初回降伏 $|\tau|=250/\sqrt3$、完全塑性 $H=0$、塑性ステップの降伏残差・塑性体積増分・平均応力保存。

基準問題は一軸**ひずみ** `deps=diag(0.003,0,0)`。
期待値は $\alpha=0.000964274423$, $q=250.964274423$ MPa、応力対角成分 $[692.309516282,441.345241859,441.345241859]$ MPa。
数値比較は例として応力 `atol=1e-7 MPa, rtol=1e-9`、ひずみ `atol=1e-11`。丸めた表示値ではなく内部値を比較する。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

In [ ]:
def j2_update(deps, sigma_n, ep_n, alpha_n, E, nu, sigma_y0, H):
    # TODO: 検証 → 弾性予測 → 分岐 → 塑性修正 → 新しい状態を返す
    raise NotImplementedError

def check_ex3():
    deps = np.diag([0.003, 0., 0.])
    stress, ep, alpha, da = j2_update(deps, Z, Z, 0., E, nu, sigma_y0, H)
    np.testing.assert_allclose(np.diag(stress),
        [692.309516282, 441.345241859, 441.345241859], atol=1e-7, rtol=1e-9)
    np.testing.assert_allclose(alpha, 0.000964274423, atol=1e-11, rtol=0)
    np.testing.assert_allclose(alpha, da, atol=1e-12)
    np.testing.assert_allclose(np.trace(ep), 0., atol=1e-12)
    np.testing.assert_allclose(stress, elastic_stress(deps-ep, E, nu), atol=1e-7)
    q = invariants(stress)[3]
    np.testing.assert_allclose(q, sigma_y0+H*alpha, atol=1e-7)
    print("応力対角成分:", np.diag(stress), "alpha:", alpha, "降伏残差:", q-sigma_y0-H*alpha)

checkpoint("演習3の一軸ひずみ", check_ex3)
# TODO: 他の負荷・H=0・入力異常の自作検証を追加

## 演習4 — 負荷・除荷・反転を一つの履歴で追う（15点）

本文3・7節。全ひずみ $\boldsymbol\varepsilon(a)=\operatorname{diag}(a,-a/2,-a/2)$、$a:0\to0.004\to-0.004\to0$。各区間40増分、初期点込み121状態とする。区間境界を重複させない。

1. 計算前に、弾性除荷中の $\alpha$、反転塑性中の $\varepsilon^p_{xx}$、全ひずみゼロへ戻ったときの応力を予想する。一軸応力試験との違いを説明する（5点）。
2. 演習3を呼ぶ履歴ドライバを自作する（6点）。各時刻の全ひずみを作り、前時刻との差を `deps` とする。応力・塑性ひずみ・$\alpha$・$\Delta\alpha$ を保存する。$\sigma_{xx}$ 対 $a$、$q$ と $\sigma_y$ 対増分番号、$\alpha$ 対増分番号の3図を描き、軸の量と単位を明示する。
3. 全ステップの $\alpha$ 非減少、降伏関数 $f\le$ 許容値、塑性ステップで $|f|$ が小さいこと、累積全ひずみからの構成則再計算を検証する。反転後に成分が減っても累積量が増す区間を特定する（4点）。

期待する性質は弾性除荷で累積量一定、塑性再負荷で増加。終点の応力ゼロは要求しない。
実装後、上の反転履歴を一本の「始点0→終点0」増分へ置換して比較し、端点情報だけでは失われるものを説明する。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

In [ ]:
# TODO: 履歴ドライバ、3図、各ステップの検証をここに実装
# 初期状態も保存する。ゼロ状態の配列は .copy() して状態を分離する。
# 未実装時に成功表示をしない。

## 演習5 — 数値更新を信頼するための検証（15点）

本文5.3・7節。検証対象を以下の四つに分ける。

1. **座標回転**（4点）：$z$ 軸回り37度の直交行列 $\mathbf Q$ を自作し、$\mathbf Q^T\mathbf Q=\mathbf I$ を確認する。ゼロから塑性負荷した状態を作り、そこからせん断を加える更新を使う。全入力テンソルを $\mathbf Q^T\mathbf A\mathbf Q$ へ変換して更新し、出力応力と塑性ひずみが同じ規則で変換され、$\alpha$ が不変なことを検証する。応力誤差のFrobeniusノルムを表示する。
2. **入力履歴の保存**（3点）：塑性履歴が非ゼロの状態から同じ増分を2回独立に呼び出す。入力配列の呼出し前後の完全一致と二つの出力の一致を確認する。連続する二増分として呼ぶ実験とは区別する。
3. **非比例経路の細分化**（5点）：まず全ひずみをゼロから $\operatorname{diag}(0.004,-0.002,-0.002)$ へ、続いて対角成分を保持したまま $\gamma_{xy}=0.008$（行列成分0.004）まで増加させる。各直線区間を $N=10,20,40,80$ 分割する。$N=640$ の比較用細分解に対して、最終応力差ノルム（MPa）と最終 $\alpha$ の絶対差を表にする。
4. **解釈**（3点）：局所の降伏残差と負荷経路の離散化誤差が違う理由を説明する。$N=80$ の応力差が $N=10$ より小さくなることを確認する。細分解を厳密解と呼ばず、特定の収束次数は合格条件にしない。材料点のみなのでメッシュ収束と呼ばない。

1〜3は実装・数値検証10点、理由と結果の説明を計5点として評価する。比例単調経路だけで刻み依存がないと結論しない。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

In [ ]:
# TODO: 回転比較、入力不変性、細分化の表を実装
# 応力のノルム: np.linalg.norm(stress_coarse - stress_fine)
# 内部状態を初期化してから各 N の経路を計算する。

## 演習6 — 積分点からCAEレビューへ（10点）

本文6・8節。以下は教材用の架空データであり、実務データは不要。

| ケース | 同一積分点・同一時刻の出力 | 補足 |
| --- | --- | --- |
| A | $q=180$ MPa、$\alpha=0.02$、$\Delta\alpha=0$ | 共通定数、除荷中 |
| B | $q=270$ MPa、$\alpha=0.02$、$\Delta\alpha>0$ | 共通定数 |
| C | $q=290$ MPa、$\alpha=0.02$ | 共通定数の速度非依存・等方硬化モデルと申告 |

1. A〜Cで現在の降伏応力と $f$ を計算し、モデルと整合するかを説明する。Cの原因は断定せず、最初に照合する情報を二つ挙げる（3点）。
2. ある反復プログラムはNewton反復ごとに `alpha_n = alpha_new` とし、確定済み履歴を書き換えている。何が二重計上されるかを説明し、確定・試行状態を分けた擬似コードを書く。材料点の $f=0$ だけで全体釣合いを保証できない理由も述べる（3点）。
3. $\sigma_{zz}=0$ に上書きした3D更新を平面応力モデルとして採用できるか。平面ひずみとの違いを含め説明する。小ひずみのコードを大回転へそのまま使えない理由を、物質座標・Cauchy応力の面積基準と結び付ける（2点）。
4. LS-DYNA等の相当塑性ひずみをこの $\alpha$ と比較する前に、材料則・出力位置・履歴の三観点で確認項目を挙げる。節点平均表示だけでの検証が不足する理由も書く（2点）。

採点は材料モデル内の整合性と、データから言える限界の区別を重視する。

### 記述・数式の答案

- 導出・予測：

（ここに記入）

- 計算結果と検証の解釈：

（実行後に記入）

## 提出前チェックとフィードバック

- [ ] 6問の記述・数式・自作コード・出力を保存した。
- [ ] 再起動後の全セル実行で例外・「未実装」がない。
- [ ] 弾性・塑性・反転・静水圧・純せん断・H=0を別々に確認した。
- [ ] 応力更新の入力保存、回転、負荷経路細分化を数値で確認した。
- [ ] 学習ログに、分かったこと、つまずいた導出、検証結果、改善案を記入した。

### 学習者コメント

- 流れ則から累積量・材料更新へのつながり：
- 実装を自分で設計する余地と検証条件の具体性：
- 次回確認したいこと：